# [교육] 온라인 강의 사용자 로그 분석 및 시각화
- 사용자 로그로 온라인 강의의 ‘참여’와 ‘이탈’을 어떻게 읽을 것인가?

- 온라인 강의 데이터 기반 대시보드 구축 프로젝트
    - 학습자 참여도 진단 → 이탈 구간 발견 → 개선 방향 제시

### 주제 목표
- 사용자 로그 기반 온라인 강의 학생참여도 진단 및 수료율 개선안 제시

- 학습자 행동 로그를 기반으로 다양한 지표를 모니터링

    - 등록자는 많지만 실제 학습 시작자는 충분한가?
    - 시작한 학생들은 꾸준히 학습하는가?
    - 영상 시청, 챕터 탐색, 포럼 참여 같은 행동은 수료와 어떤 관련이 있는가?
    - 특정 코스나 특정 학습자 집단에서 이탈이 더 심한가?

In [147]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import matplotlib.pyplot as plt
import platform
import os
import glob

plt.rcParams['font.family'] = 'Malgun Gothic'

In [148]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import scipy as stats
# import seaborn as sns
# import numpy as np

# 출력 짤림 방지
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [149]:
# 데이터 불러오기
df = pd.read_csv('./data/Courses.csv', parse_dates=['start_time_DI', 'last_event_DI'])
df.head()


,index,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
0,0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,NaN,1.0
1,1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,NaN,0,2012-10-15,NaT,NaN,9.0,NaN,1.0,0,NaN,1.0
2,2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,NaN,1.0
3,3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-09-17,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0
4,4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0


In [150]:
# df = pd.read_csv("data/Courses.csv")

In [151]:
# 원본데이터 결측치 개수, 비율
display(pd.DataFrame({
    'sum': df.isna().sum(),
    'ratio': df.isna().mean() * 100
}).sort_values('ratio', ascending=False).reset_index())

,index,sum,ratio
0,roles,641138,100.000000
1,incomplete_flag,540977,84.377622
2,nplay_video,457530,71.362172
3,nchapters,258753,40.358394
4,nevents,199151,31.062111
5,last_event_DI,178954,27.911932
6,ndays_act,162743,25.383459
7,LoE_DI,106008,16.534350
8,YoB,96605,15.067739
9,gender,86806,13.539363


In [152]:
# 전처리용 데이터 셋 생성
pre = df.copy()

In [153]:
# 중복행 확인
len(pre.duplicated())
pre.duplicated().sum()

np.int64(0)

In [154]:
# 의미없는 컬럼 제거
pre = pre.drop(columns=['index', 'roles'])


# 데이터 확인

In [155]:
pre.head()

,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,incomplete_flag
0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,1.0
1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,NaN,0,2012-10-15,NaT,NaN,9.0,NaN,1.0,0,1.0
2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,1.0
3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-09-17,NaT,NaN,16.0,NaN,NaN,0,1.0
4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,NaT,NaN,16.0,NaN,NaN,0,1.0


In [156]:
pre.shape

(641138, 19)

In [157]:
pre.info()

<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   course_id          641138 non-null  str           
 1   userid_DI          641138 non-null  str           
 2   registered         641138 non-null  int64         
 3   viewed             641138 non-null  int64         
 4   explored           641138 non-null  int64         
 5   certified          641138 non-null  int64         
 6   final_cc_cname_DI  641138 non-null  str           
 7   LoE_DI             535130 non-null  str           
 8   YoB                544533 non-null  float64       
 9   gender             554332 non-null  str           
 10  grade              592766 non-null  str           
 11  start_time_DI      641138 non-null  datetime64[us]
 12  last_event_DI      462184 non-null  datetime64[us]
 13  nevents            441987 non-null  float64       
 14 

In [158]:
pre.isna().sum()

course_id                 0
userid_DI                 0
registered                0
viewed                    0
explored                  0
certified                 0
final_cc_cname_DI         0
LoE_DI               106008
YoB                   96605
gender                86806
grade                 48372
start_time_DI             0
last_event_DI        178954
nevents              199151
ndays_act            162743
nplay_video          457530
nchapters            258753
nforum_posts              0
incomplete_flag      540977
dtype: int64

# 주제 확인

### 중요 컬럼명
- **registered**: 강의 등록 여부
- **viewed**: 강의를 실제로 열어본 적이 있는지
- **explored**: 강의 콘텐츠를 더 깊게 탐색했는지
- **certified**: 최종적으로 인증서를 획득했는지
- **grade**: 학습 성취 수준
- **nevents**: 전체 학습 이벤트 수
- **ndays_act**: 실제 활동한 일수
- **nplay_video**: 영상 재생 수
- **nchapters**: 탐색한 챕터 수
- **nforum_posts**: 포럼 활동량


-> 학습자의 행동을 다음과 같은 퍼널로 해석할 수 있다

    - 등록 → 첫 방문 → 적극적 탐색 → 지속 학습 → 인증/수료

### 핵심 질문
- 우리 플랫폼의 전체 학습 퍼널은 어떻게 생겼는가?
- 학생들은 **어느 단계에서 가장 많이 이탈**하는가?
- **참여도가 높은 학생과 낮은 학생**은 어떤 행동 차이를 보이는가?
- 어떤 코스가 상대적으로 **참여 유지율 / 인증 전환율**이 높은가?
- 국가, 성별, 학력, 연령대 같은 특성에 따라 차이가 보이는가?
- 영상 재생 수, 활동일수, 챕터 탐색 수, 포럼 활동은 성취도나 인증과 어떤 관련이 있는가?
- 우리 서비스가 학생참여도를 높이기 위해 우선적으로 개선해야 할 지점은 무엇인가?

### 대시보드 작성 프로세스
- 분석 기획 완료 후 분석 주제 세분화
    - 학습자 관점 / 코스 관점 / 퍼널 관점
- 각 주제별로 어떤 질문에 답할지 정의
- 노트에 먼저 손으로 스케치
    - KPI 카드 위치
    - 필터 위치
    - 차트 간 연결 흐름
    - 사용자가 어떤 순서로 읽을지
- 스케치 후 워크시트 제작을 진행
- 제작 중 가시성이 떨어지거나 메시지가 약하면 과감히 수정

# 데이터 딕셔너리 

In [159]:
display(pre.describe(include='all').T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
course_id,641138,16,HarvardX/CS50x/2012,169621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
userid_DI,641138,476532,MHxPC130505428,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registered,641138.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,0.0
viewed,641138.0,NaN,NaN,NaN,0.624299,0.0,0.0,1.0,1.0,1.0,0.484304
explored,641138.0,NaN,NaN,NaN,0.061899,0.0,0.0,0.0,0.0,1.0,0.240973
certified,641138.0,NaN,NaN,NaN,0.027587,0.0,0.0,0.0,0.0,1.0,0.163786
final_cc_cname_DI,641138,34,United States,184240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LoE_DI,535130,5,Bachelor's,219768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YoB,544533.0,NaN,NaN,NaN,1985.253279,1931.0,1982.0,1988.0,1991.0,2013.0,8.891814
gender,554332,3,m,411520,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [160]:
pre[['incomplete_flag']].value_counts()

incomplete_flag
1.0                100161
Name: count, dtype: int64

In [161]:
print(pre['incomplete_flag'].dtypes)
print(pre['incomplete_flag'].isnull().sum())
print((pre['incomplete_flag'].isnull().mean() * 100).round(2))
print((pre['incomplete_flag'].isnull().mean() * 100))

float64
540977
84.38
84.37762229036494


# 주제 확인

### 중요 컬럼명
- **registered**: 강의 등록 여부
- **viewed**: 강의를 실제로 열어본 적이 있는지
- **explored**: 강의 콘텐츠를 더 깊게 탐색했는지
- **certified**: 최종적으로 인증서를 획득했는지
- **grade**: 학습 성취 수준
- **nevents**: 전체 학습 이벤트 수
- **ndays_act**: 실제 활동한 일수
- **nplay_video**: 영상 재생 수
- **nchapters**: 탐색한 챕터 수
- **nforum_posts**: 포럼 활동량


-> 학습자의 행동을 다음과 같은 퍼널로 해석할 수 있다

    - 등록 → 첫 방문 → 적극적 탐색 → 지속 학습 → 인증/수료

### 핵심 질문
- 우리 플랫폼의 전체 학습 퍼널은 어떻게 생겼는가?
- 학생들은 **어느 단계에서 가장 많이 이탈**하는가?
- **참여도가 높은 학생과 낮은 학생**은 어떤 행동 차이를 보이는가?
- 어떤 코스가 상대적으로 **참여 유지율 / 인증 전환율**이 높은가?
- 국가, 성별, 학력, 연령대 같은 특성에 따라 차이가 보이는가?
- 영상 재생 수, 활동일수, 챕터 탐색 수, 포럼 활동은 성취도나 인증과 어떤 관련이 있는가?
- 우리 서비스가 학생참여도를 높이기 위해 우선적으로 개선해야 할 지점은 무엇인가?

### 대시보드 작성 프로세스
- 분석 기획 완료 후 분석 주제 세분화
    - 학습자 관점 / 코스 관점 / 퍼널 관점
- 각 주제별로 어떤 질문에 답할지 정의
- 노트에 먼저 손으로 스케치
    - KPI 카드 위치
    - 필터 위치
    - 차트 간 연결 흐름
    - 사용자가 어떤 순서로 읽을지
- 스케치 후 워크시트 제작을 진행
- 제작 중 가시성이 떨어지거나 메시지가 약하면 과감히 수정

# 데이터 딕셔너리 

In [162]:
display(df.describe(include='all').T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
index,641138.0,NaN,NaN,NaN,320568.5,0.0,160284.25,320568.5,480852.75,641137.0,185080.742781
course_id,641138,16,HarvardX/CS50x/2012,169621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
userid_DI,641138,476532,MHxPC130505428,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registered,641138.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,0.0
viewed,641138.0,NaN,NaN,NaN,0.624299,0.0,0.0,1.0,1.0,1.0,0.484304
explored,641138.0,NaN,NaN,NaN,0.061899,0.0,0.0,0.0,0.0,1.0,0.240973
certified,641138.0,NaN,NaN,NaN,0.027587,0.0,0.0,0.0,0.0,1.0,0.163786
final_cc_cname_DI,641138,34,United States,184240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LoE_DI,535130,5,Bachelor's,219768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YoB,544533.0,NaN,NaN,NaN,1985.253279,1931.0,1982.0,1988.0,1991.0,2013.0,8.891814


In [163]:
df[['incomplete_flag']].value_counts()

incomplete_flag
1.0                100161
Name: count, dtype: int64

In [164]:
print(df['incomplete_flag'].dtypes)
print(df['incomplete_flag'].isnull().sum())
print((df['incomplete_flag'].isnull().mean() * 100).round(2))
print((df['incomplete_flag'].isnull().mean() * 100))

float64
540977
84.38
84.37762229036494


# 전처리 통합

In [165]:
### 결측치 대체 & 행 제거

In [166]:
# course_id 이상치 
    # 확인 후 이상치의 공통점 찾아서 처리여부 파악 (직접 찍어보고 확인)
    
pre[['course_code', 'course', 'course_year']] = pre['course_id'].str.split('/', expand=True) # expand=True df로 반환(열로세우기)

# CS50x는 분류 상 '상시 강의' / '기간 강의' 중 본 프로젝트의 목표에 맞지 않는 분류라 분석에 사용하지 않기로 결정
pre = pre[~pre['course_id'].str.contains('CS50x')].copy()   #~로 아닌 애들만 ㄱㄱ

print(pre['course'].value_counts())

print(pre['course_year'].value_counts())

course
6.00x     124446
6.002x     63046
ER22x      57406
PH207x     41592
PH278x     39602
8.02x      31048
CB22x      30002
14.73x     27870
7.00x      21009
3.091x     20354
8.MReV      9477
2.01x       5665
Name: count, dtype: int64
course_year
2013_Spring    298691
2012_Fall      163349
2013_Summer      9477
Name: count, dtype: int64


In [167]:
# 나라 컬럼명 변경
pre = pre.rename(columns={'final_cc_cname_DI':'country'})

In [168]:
print(pre['nevents'].value_counts().sort_index())
print(pre['ndays_act'].value_counts().sort_index())
print(pre['nchapters'].value_counts().sort_index())

nevents
1.0         56670
2.0         30867
3.0         15397
4.0          9808
5.0          7478
6.0          6374
7.0          5533
8.0          5038
9.0          4748
10.0         4372
11.0         4113
12.0         3818
13.0         3571
14.0         3360
15.0         3160
16.0         3084
17.0         2851
18.0         2734
19.0         2614
20.0         2554
21.0         2433
22.0         2311
23.0         2201
24.0         2181
25.0         2047
26.0         2016
27.0         1882
28.0         1836
29.0         1803
30.0         1768
31.0         1654
32.0         1569
33.0         1620
34.0         1528
35.0         1531
36.0         1471
37.0         1464
38.0         1390
39.0         1350
40.0         1306
41.0         1302
42.0         1223
43.0         1262
44.0         1151
45.0         1175
46.0         1151
47.0         1081
48.0         1099
49.0         1029
50.0         1072
51.0         1021
52.0         1034
53.0          988
54.0          956
55.0          958
56

In [169]:
# n~ 컬럼
n_cols = ['nplay_video', 'nevents', 'ndays_act', 'nchapters']
pre[n_cols] = pre[n_cols].fillna(0)
# 근거 : nplay_video, 0이 존재하지 않고, '시청시간이 없는 값'이 모두 NaN 처리가 되었다는 논리적 근거 O
# 데이터 상 결측이 존재하나 값 중 0이 존재하지 않고 / 시스템 상에서 활동이 없는 경우(의미적 0)를 NaN으로 남긴 것이다.
# 따라서 본 데이터 분석에서는 NaN를 '활동하지 않음'(=0)으로 간주하여 분석을 진행하였다. 

In [170]:
# 110531 <<< 제거대상 
pre = pre.drop(pre[(pre['viewed'] == 0) & ((pre['nchapters'] > 0) | (pre['nevents'] > 0))].index)


In [171]:
# 논리적 이상치
    # nchapters값이 1인데 nan값으로 입력되었을 리 X ∴ 이상치
    # nplay_video의 기준이 클릭/완강인지 명시 X, 
    # But viewed 컬럼이 따로 존재하는 것으로 nplay_video가 완강 기준일 것이라는 근거로 분석을 진행했다.
    # nplay_video가 완강 기준일 것이라는 근거로 분석을 진행했다.
nonono = (pre['viewed'] == 0) & ((pre['nchapters'] > 0) | (pre['nevents'] > 0))

In [172]:
# start_time_DI (결측 71.36) -> 대신 ndays_act 이용하기?
# 음수가 나오는 애들 제거 후 한계점 감안하고 사용하라는 튜터님 피드백 있었음


In [173]:
# 아니면 ndays_act를 이용하는 방법 (몇일 이상 접속한 사람으로 구간 나눠서?) -> 그 기준은?
# 조사 필요할 듯
# ndays_act를 가지고 몇일 이상 접속해야 찍먹러가 아닌 기준에 대해서 명확하게 하고 싶은데

In [174]:
# 찍먹과 학습의지 수강생을 나눌 수 있을까?

ac_df = df[df['viewed'] == 1].copy()
viewed_rate = (df['viewed'] == 1).mean() * 100
viewed_rate # 62.42

np.float64(62.429929281995456)

In [175]:
# 파생컬럼 생성

# 학생들의 나이(age)
pre['age'] = pre['start_time_DI'].dt.year - pre['YoB']

# YoB 결측치를 0으로 채우기
pre['YoB'] = pre['YoB'].fillna(0)

# 최소 가입 나이 (13세)
pre.loc[pre['age'] < 13, 'age']

# 퍼널 단계 컬럼(step): 각 학생 별 진행 단계
pre['step'] = np.select(
    [
        pre['certified'] ==1,
        pre['explored'] == 1,
        pre['viewed'] == 1,
        pre['registered'] == 1,
    ],
    [
        'Certified',
        'Explored',
        'Viewed',
        'Registered'
    ],
    default='None'
)

In [176]:
# Unknown
un_cols = ['gender', 'LoE_DI', 'YoB']
pre[un_cols] = pre[un_cols].fillna('Unknown')

In [177]:
pre['age'].describe()

count    299277.000000
mean         27.279991
std           8.893779
min           0.000000
25%          21.000000
50%          25.000000
75%          30.000000
max          82.000000
Name: age, dtype: float64

In [178]:
# 퍼널 분석을 위한 탐색
    # 결국 우리 분석 대상은
    # 이탈자 / 반복학습생이 아닐까?
    # '끝까지 수료할 가능성'이 있는 사람들은 끌어모으고 << 잠재고객? 가치 파악? 수익성 ↑?
    # '이탈할 가능성' = 리스크는 줄이는 게 맞으니까 << 사실 이게 더 중요하긴 함 : 놓친 사람
    
# 이탈 이유가 결국 수강 의지가 있는 부류 / 없는데 이탈하는 부류 < 이 문제점을 파악해야
# 등록 - 첫수강에서 막히는지(진입 or 환경) / 수강 - 유지에서 막히는지(난이도)
# 찍먹러인지 중도포기인지 반복학습자? 인지
# 근데 이탈하면 어디서 많이 이탈하는지가 중요 -> 그래서 우리가 퍼널 별 구간 이탈자 탐색 < 이걸 하는거고 음음 굿

# 찍먹러는 ndays_act가 적은 사람으로 판단 가능? < 조사 필요
# 중도포기는 certified로 잡고 싶었는데 상당수가 수료 못해서 분류 기준이 비상이고
# 

In [179]:
### 파생변수 아이디어

# 찍먹러
pre['out_cus'] = (pre['ndays_act'] <= 3)

pre['out_cus'].value_counts(normalize=True) * 100
# 2일 기준
# 찍먹러 54.87
# 논찍먹러 45.12

# 왜 이렇게 많지요...??
# 온라인이라 그냥 일단 신청하고 보자 일까

# 음 3일은 64 / 35
# 4일은 70 / 29

# 구간별로 잘라서 ~일로 열정수강생이랑 찍먹러랑 일반수강생이랑 나누고 싶은데 나누는 기준은 무엇으로?
# 이것도 추가 조사 필요함 근데 일단 찍어보니까 찍먹러는 3일 기준이 낫지 않을까 싶음 

# 코스 별로도 다를 테고, 페이지 클릭 횟수/완강 여부랑 같이 봐도 좋을 것 같음 (월요일에 추가조사+나눠서 보기 진행하면 어떨까)

out_cus
True     64.457347
False    35.542653
Name: proportion, dtype: float64

In [180]:
# 추가 조사 필요한 부분 조사
# 데이터 딕셔너리 업데이트
# 피드백 정리(문튜터님/제연튜터님) 

### 